In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from scripts.utils import Config

config = Config()
pod    = config.pod
print('bins    :', pod['bins'])
print('fit     :', pod['fit'])
print('runs    :', list(pod['runs'].keys()))

## 1. Load data

In [ ]:
import os

def load(splitsdir, coarse=True):
    prefix = '' if coarse else 'OLD_'
    dslist = []
    for splitname in ('train', 'valid'):
        filepath = os.path.join(splitsdir, f'{prefix}{splitname}.h5')
        ds = xr.open_dataset(filepath, engine='h5netcdf')[['bl', 'pr', 'lf']]
        dslist.append(ds)
    lf = dslist[0]['lf'].load()
    trainds = xr.concat(dslist, dim='time')
    bl = trainds['bl'].load()
    pr = trainds['pr'].load()
    return bl, pr, lf

# pod_bl uses coarse=True (name contains 'coarse'), pod_bl uses coarse=False
# The failing run is pod_bl -> coarse=False (no 'coarse' in name)
bl, pr, lf = load(config.splitsdir, coarse=False)
print('bl shape:', bl.shape, '  pr shape:', pr.shape)
print('bl range: [{:.4f}, {:.4f}]'.format(float(bl.min()), float(bl.max())))
print('pr range: [{:.4f}, {:.4f}]'.format(float(pr.min()), float(pr.max())))

## 2. Check finite counts and NaN fraction

In [ ]:
xflat = bl.values.ravel()
yflat = pr.values.ravel()

finite = np.isfinite(xflat) & np.isfinite(yflat)
print(f'Total samples   : {finite.size:,}')
print(f'Finite samples  : {finite.sum():,}  ({100*finite.mean():.1f}%)')
print(f'NaN in bl       : {(~np.isfinite(xflat)).sum():,}')
print(f'NaN in pr       : {(~np.isfinite(yflat)).sum():,}')

x = xflat[finite]
y = yflat[finite]
print(f'\nbl finite range : [{x.min():.4f}, {x.max():.4f}]')
print(f'pr finite range : [{y.min():.4f}, {y.max():.4f}]')

## 3. Reproduce the binning step

In [ ]:
bins        = pod['bins']
fitparams   = pod['fit']
samplethresh = bins['minsample']
prmin       = fitparams['prmin']
prmax       = fitparams['prmax']

binedges   = np.arange(bins['min'], bins['max'] + bins['width'], bins['width'])
bincenters = 0.5 * (binedges[:-1] + binedges[1:])
print(f'Bins: {bincenters.size} bins from {bincenters[0]:.4f} to {bincenters[-1]:.4f}')

binidxs = np.digitize(x, binedges) - 1
inrange = (binidxs >= 0) & (binidxs < bincenters.size)
print(f'Samples falling within bin range: {inrange.sum():,} / {inrange.size:,}')

counts = np.bincount(binidxs[inrange], minlength=bincenters.size).astype(np.int64)
sums   = np.bincount(binidxs[inrange], weights=y[inrange], minlength=bincenters.size).astype(np.float32)
with np.errstate(divide='ignore', invalid='ignore'):
    ymeans = sums / counts
ymeans[counts < samplethresh] = np.nan

n_populated = (counts >= samplethresh).sum()
n_finite    = np.isfinite(ymeans).sum()
n_in_prrange = (np.isfinite(ymeans) & (ymeans >= prmin) & (ymeans <= prmax)).sum()
print(f'\nBins with >= {samplethresh} samples : {n_populated}')
print(f'Bins with finite ymean      : {n_finite}')
print(f'Bins in prmin/prmax range   : {n_in_prrange}  <-- must be > 0 for polyfit')
print(f'\npmean of finite bins: {np.nanmean(ymeans):.4f}')
print(f'prmin={prmin}, prmax={prmax}')

## 4. Visualise binned means and fit range

In [ ]:
fitrange = np.isfinite(ymeans) & (ymeans >= prmin) & (ymeans <= prmax)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: bin counts
ax = axes[0]
ax.bar(bincenters, counts, width=bins['width'], color='steelblue', alpha=0.7)
ax.axhline(samplethresh, color='red', linestyle='--', label=f'minsample={samplethresh}')
ax.set_xlabel('BL')
ax.set_ylabel('Sample count')
ax.set_title('Bin counts')
ax.legend()

# Right: binned pr means with fit range highlighted
ax = axes[1]
ax.plot(bincenters, ymeans, color='steelblue', lw=1, label='Binned pr mean')
ax.scatter(bincenters[fitrange], ymeans[fitrange], color='orange', zorder=5,
           label=f'In fit range [{prmin}, {prmax}]')
ax.axhline(prmin, color='green', linestyle='--', lw=0.8, label=f'prmin={prmin}')
ax.axhline(prmax, color='red',   linestyle='--', lw=0.8, label=f'prmax={prmax}')
ax.set_xlabel('BL')
ax.set_ylabel('Mean pr')
ax.set_title('Binned pr means')
ax.legend()

plt.tight_layout()
plt.show()

## 5. Histogram of pr values (check units / scale)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(y[y > 0], bins=100, color='steelblue', alpha=0.8)
ax.axvline(prmin, color='green', linestyle='--', label=f'prmin={prmin}')
ax.axvline(prmax, color='red',   linestyle='--', label=f'prmax={prmax}')
ax.set_xlabel('pr (rainy samples only)')
ax.set_ylabel('Count')
ax.set_title('pr distribution (linear scale)')
ax.legend()

ax = axes[1]
ax.hist(y[y > 0], bins=100, color='steelblue', alpha=0.8)
ax.axvline(prmin, color='green', linestyle='--', label=f'prmin={prmin}')
ax.axvline(prmax, color='red',   linestyle='--', label=f'prmax={prmax}')
ax.set_yscale('log')
ax.set_xlabel('pr (rainy samples only)')
ax.set_ylabel('Count (log)')
ax.set_title('pr distribution (log scale)')
ax.legend()

plt.tight_layout()
plt.show()

print(f'pr percentiles:')
for p in [50, 75, 90, 95, 99, 99.9]:
    print(f'  {p:5.1f}th: {np.percentile(y[y>0], p):.4f}')